# 12 — Seed stability of the transformer results

Across three runs of an identical protocol on the same train/test split, four
different models have held first place — SciBERT, BioLinkBERT, Bio+ClinicalBERT
and BiomedBERT-large — all separated by one to three errors out of 341.

Because the split never changed, that movement is not split variance. It is
**training stochasticity**: weight initialisation, batch ordering, and
non-deterministic cuDNN kernels. This notebook measures it directly, by refitting
each model's selected configuration under several torch seeds and reporting the
spread of test-set results.

## Why this rather than repeated train/test splits

Repeated splits (~15 h) would measure a different quantity — sensitivity to the
partition — and would not answer the question the results actually raise. The
instability observed here appeared *within* a fixed partition, so seeds are the
variable to vary. This costs about 40 minutes.

Together with two measurements we already have, uncertainty is then characterised
from three independent directions:

| Source | Measured by |
|---|---|
| Test-sample variation | bootstrap CIs over test records (notebook 07) |
| Train-split variation | standard deviation across the 5 CV folds (notebook 05) |
| **Training stochasticity** | **this notebook** |

Nothing here touches model selection: every configuration is fixed, read from
the completed run. The test set is used only to score already-chosen models.

In [1]:
import os, sys, json, time, copy, warnings, glob
from pathlib import Path
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np, pandas as pd, torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score

RUN_DIR = Path(os.environ.get("BMO_RUN_DIR", "/mnt/subedi-research/projects/bmo_rerun_2026"))
sys.path.insert(0, str(RUN_DIR))
import bmo_common as C

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import torch._dynamo; torch._dynamo.config.suppress_errors = True

GPU_CAP_GB = float(os.environ.get("BMO_GPU_CAP_GB", "10"))
if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    torch.cuda.set_per_process_memory_fraction(min(0.95, GPU_CAP_GB/total), 0)
    print(f"device {torch.cuda.get_device_name(0)} | VRAM cap {GPU_CAP_GB:.0f} GB")

SEEDS = [int(x) for x in os.environ.get("BMO_SEEDS", "42,1337,2024,7,99").split(",")]
if os.environ.get("BMO_SMOKE") == "1":
    SEEDS = SEEDS[:2]
    C.OUT_BERT = C.OUT_BERT.parent / "bert_models_smoke"
    C.OUT_BERT.mkdir(parents=True, exist_ok=True)
print("seeds:", SEEDS)

device NVIDIA L40 | VRAM cap 10 GB
seeds: [42, 1337, 2024, 7, 99]


## Recover each model's selected configuration

Read from the completed run rather than re-derived, so this measures the
variability of the *chosen* models and cannot accidentally re-select anything.

In [2]:
CKPT = {
    "BERT-base": "bert-base-uncased",
    "BioBERT": "dmis-lab/biobert-v1.1",
    "PubMedBERT": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
    "Bio+ClinicalBERT": "emilyalsentzer/Bio_ClinicalBERT",
    "BioLinkBERT": "michiyasunaga/BioLinkBERT-base",
    "SciBERT": "allenai/scibert_scivocab_uncased",
    "DeBERTa-v3": "microsoft/deberta-v3-base",
    "ModernBERT": "answerdotai/ModernBERT-base",
    "BiomedBERT-large": "microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract",
}
src = sorted((C.OUT_BERT.parent / "bert_models").glob("test_results_shard*.csv"))
cfg_df = pd.concat([pd.read_csv(f) for f in src], ignore_index=True)
need = ["Model", "lr", "batch_size", "epochs", "dropout", "weight_decay", "errors"]
CONFIGS = {r["Model"]: r for _, r in cfg_df.iterrows() if r["Model"] in CKPT}
print(cfg_df[[c for c in need if c in cfg_df.columns]].round(5).to_string(index=False))
print(f"\n{len(CONFIGS)} configurations recovered")

           Model      lr  batch_size  epochs  dropout  weight_decay  errors
     BioLinkBERT 0.00005          32       2      0.3          0.01       1
BiomedBERT-large 0.00002           8       5      0.5          0.10       2
       BERT-base 0.00003          16       5      0.5          0.10       3
      DeBERTa-v3 0.00002          16       7      0.1          0.10       3
      PubMedBERT 0.00003          16       1      0.1          0.01       5
Bio+ClinicalBERT 0.00005          32       5      0.5          0.01       3
         SciBERT 0.00003          16       3      0.5          0.01       4
      ModernBERT 0.00005          32       6      0.3          0.01       5
         BioBERT 0.00002          32       2      0.1          0.10       7

9 configurations recovered


In [3]:
MAX_LEN, WARMUP_RATIO, LR_LAYER_DECAY = 128, 0.1, 0.95
df, train_idx, test_idx = C.load_data()
texts_all = df["clean_freeform"].fillna("").values
labels_all = df["has_biomarker"].values
texts, labels = texts_all[train_idx], labels_all[train_idx]
texts_te, labels_te = texts_all[test_idx], labels_all[test_idx]
print(f"train {len(texts):,} | test {len(texts_te):,}")

class DS(Dataset):
    def __init__(self, t, y, tok): self.t, self.y, self.tok = t, y, tok
    def __len__(self): return len(self.t)
    def __getitem__(self, i):
        e = self.tok(str(self.t[i]), max_length=MAX_LEN, truncation=True,
                     padding="max_length", return_tensors="pt")
        return {"input_ids": e["input_ids"].squeeze(0),
                "attention_mask": e["attention_mask"].squeeze(0),
                "label": torch.tensor(int(self.y[i]), dtype=torch.long)}

class Clf(nn.Module):
    def __init__(self, name, dropout):
        super().__init__()
        cfg = AutoConfig.from_pretrained(name)
        if hasattr(cfg, "reference_compile"):
            cfg.reference_compile = False       # needs CPython headers this node lacks
        self.bert = AutoModel.from_pretrained(name, config=cfg)
        self.head = nn.Sequential(nn.Dropout(dropout),
                                  nn.Linear(self.bert.config.hidden_size, 2))
    def forward(self, ids, mask):
        return self.head(self.bert(input_ids=ids, attention_mask=mask).last_hidden_state[:, 0, :])

def layers_of(b):
    for path in (("encoder", "layer"), ("layers",), ("encoder", "layers")):
        o = b
        for a in path:
            o = getattr(o, a, None)
            if o is None: break
        if o is not None and hasattr(o, "__len__") and len(o): return o
    raise AttributeError(type(b).__name__)

def make_opt(m, lr, wd):
    nd = ("bias", "LayerNorm.weight", "layer_norm.weight", "norm.weight")
    groups = []
    def add(named, l):
        named = list(named)
        d = [p for n, p in named if not any(k in n for k in nd) and p.requires_grad]
        x = [p for n, p in named if any(k in n for k in nd) and p.requires_grad]
        if d: groups.append({"params": d, "lr": l, "weight_decay": wd})
        if x: groups.append({"params": x, "lr": l, "weight_decay": 0.0})
    L = layers_of(m.bert); n = len(L)
    add(m.head.named_parameters(), lr)
    for i in range(n - 1, -1, -1):
        add(L[i].named_parameters(), lr * (LR_LAYER_DECAY ** (n - i)))
    emb = getattr(m.bert, "embeddings", None)
    if emb is not None: add(emb.named_parameters(), lr * (LR_LAYER_DECAY ** (n + 1)))
    return AdamW(groups)

train 1,559 | test 341


## Refit each model once per seed

Only the seed changes. Configuration, data, split and epoch budget are fixed, so
the spread of results isolates training stochasticity.

In [4]:
def fit_and_score(name, ckpt, lr, bs, epochs, dropout, wd, seed):
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    tok = AutoTokenizer.from_pretrained(ckpt)
    m = Clf(ckpt, dropout).to(DEVICE)
    tl = DataLoader(DS(texts, labels, tok), batch_size=bs, shuffle=True,
                    num_workers=2, pin_memory=True,
                    generator=torch.Generator().manual_seed(seed))
    vl = DataLoader(DS(texts_te, labels_te, tok), batch_size=max(bs, 64),
                    shuffle=False, num_workers=2, pin_memory=True)
    opt = make_opt(m, lr, wd)
    steps = max(1, len(tl)) * epochs
    sch = get_linear_schedule_with_warmup(opt, int(WARMUP_RATIO * steps), steps)
    neg, pos = int((labels == 0).sum()), int((labels == 1).sum())
    crit = nn.CrossEntropyLoss(weight=torch.tensor([1.0, neg/max(pos,1)],
                                                   dtype=torch.float, device=DEVICE))
    m.train()
    for _ in range(epochs):
        for b in tl:
            opt.zero_grad(set_to_none=True)
            loss = crit(m(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE)),
                        b["label"].to(DEVICE))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step(); sch.step()
    m.eval(); preds, probs = [], []
    with torch.no_grad():
        for b in vl:
            p = torch.softmax(m(b["input_ids"].to(DEVICE),
                                b["attention_mask"].to(DEVICE)).float(), 1)[:, 1]
            probs.extend(p.cpu().tolist()); preds.extend(p.ge(0.5).long().cpu().tolist())
    del m; torch.cuda.empty_cache()
    return np.array(preds), np.array(probs)

SHARD = os.environ.get("BMO_SHARD", "all")
models = list(CONFIGS)
if SHARD != "all":
    k, n = int(SHARD), int(os.environ.get("BMO_NSHARD", "2"))
    models = [m for i, m in enumerate(models) if i % n == k]
print("this process handles:", models)

rows = []
for name in models:
    c = CONFIGS[name]
    lr, bs = float(c["lr"]), int(c["batch_size"])
    ep = int(c.get("epochs", 4) or 4)
    dp = float(c.get("dropout", 0.3) or 0.3)
    wd = float(c.get("weight_decay", 0.01) or 0.01)
    print(f"\n{'='*70}\n{name}  lr={lr:g} bs={bs} epochs={ep} dropout={dp} wd={wd}\n{'='*70}")
    for sd in SEEDS:
        t0 = time.time()
        yp, pp = fit_and_score(name, CKPT[name], lr, bs, ep, dp, wd, sd)
        r = C.evaluate(labels_te, yp, pp, name=name, tier="Transformer",
                       extra={"seed": sd, "seconds": round(time.time()-t0, 1)})
        rows.append(r)
        print(f"  seed {sd:<6d} weighted F1 {r['weighted_f1']:.4f} | "
              f"errors {r['errors']:<3d} | FP {r['FP']} FN {r['FN']} | {r['seconds']:.0f}s")

seed_df = pd.DataFrame(rows)
C.save_table(seed_df, C.OUT_BERT / f"seed_stability_shard{SHARD}.csv")

this process handles: ['BiomedBERT-large', 'DeBERTa-v3', 'Bio+ClinicalBERT', 'ModernBERT']

BiomedBERT-large  lr=2e-05 bs=8 epochs=5 dropout=0.5 wd=0.1


  seed 42     weighted F1 0.9853 | errors 5   | FP 2 FN 3 | 108s


  seed 1337   weighted F1 0.9853 | errors 5   | FP 2 FN 3 | 109s


  seed 2024   weighted F1 0.9941 | errors 2   | FP 2 FN 0 | 110s


  seed 7      weighted F1 0.9941 | errors 2   | FP 1 FN 1 | 110s


  seed 99     weighted F1 0.9853 | errors 5   | FP 2 FN 3 | 110s

DeBERTa-v3  lr=2e-05 bs=16 epochs=7 dropout=0.1 wd=0.1


... (12 outputs left out)

  seed 2024   weighted F1 0.9852 | errors 5   | FP 0 FN 5 | 45s


  seed 7      weighted F1 0.9793 | errors 7   | FP 1 FN 6 | 45s


  seed 99     weighted F1 0.9882 | errors 4   | FP 0 FN 4 | 45s
table  -> /mnt/subedi-research/projects/bmo_rerun_2026/outputs/bert_models/seed_stability_shard1.csv  (20 rows)


,Model,Tier,accuracy,balanced_acc,weighted_f1,macro_f1,biomarker_f1,precision,recall,specificity,...,FP,FN,TP,errors,n,auc,auprc,brier,seed,seconds
0,BiomedBERT-large,Transformer,0.985337,0.981928,0.985319,0.983103,0.976959,0.981481,0.972477,0.991379,...,2,3,106,5,341,0.998853,0.997295,0.011738,42,107.8
1,BiomedBERT-large,Transformer,0.985337,0.981928,0.985319,0.983103,0.976959,0.981481,0.972477,0.991379,...,2,3,106,5,341,0.998418,0.996350,0.009744,1337,108.9
2,BiomedBERT-large,Transformer,0.994135,0.995690,0.994149,0.993290,0.990909,0.981982,1.000000,0.991379,...,2,0,109,2,341,0.999209,0.998189,0.005371,2024,109.5
3,BiomedBERT-large,Transformer,0.994135,0.993258,0.994135,0.993258,0.990826,0.990826,0.990826,0.995690,...,1,1,108,2,341,0.999328,0.998468,0.004590,7,109.9
4,BiomedBERT-large,Transformer,0.985337,0.981928,0.985319,0.983103,0.976959,0.981481,0.972477,0.991379,...,2,3,106,5,341,0.998675,0.997013,0.014205,99,109.8
5,DeBERTa-v3,Transformer,0.979472,0.972754,0.979395,0.976226,0.967442,0.981132,0.954128,0.991379,...,2,5,104,7,341,0.994938,0.992753,0.020593,42,62.1
6,DeBERTa-v3,Transformer,0.985337,0.977064,0.985243,0.982932,0.976526,1.000000,0.954128,1.000000,...,0,5,104,5,341,0.995057,0.992020,0.014930,1337,61.8
7,DeBERTa-v3,Transformer,0.982405,0.974909,0.982315,0.979571,0.971963,0.990476,0.954128,0.995690,...,1,5,104,6,341,0.997232,0.995352,0.016706,2024,61.8
8,DeBERTa-v3,Transformer,0.988270,0.981651,0.988210,0.986381,0.981308,1.000000,0.963303,1.000000,...,0,4,105,4,341,0.998181,0.996783,0.012056,7,62.0
9,DeBERTa-v3,Transformer,0.979472,0.970322,0.979341,0.976105,0.967136,0.990385,0.944954,0.995690,...,1,6,103,7,341,0.996441,0.994919,0.018252,99,62.1


## Spread across seeds

`errors_sd` is the number that answers a reviewer asking how reproducible the
ranking is. If it approaches the gap between adjacent systems in the main table,
the ordering between them is not a real distinction.

In [5]:
if len(seed_df):
    agg = (seed_df.groupby("Model")
           .agg(n_seeds=("seed", "size"),
                errors_mean=("errors", "mean"), errors_sd=("errors", "std"),
                errors_min=("errors", "min"), errors_max=("errors", "max"),
                wf1_mean=("weighted_f1", "mean"), wf1_sd=("weighted_f1", "std"))
           .round(4).sort_values("errors_mean").reset_index())
    print(agg.to_string(index=False))
    C.save_table(agg, C.OUT_BERT / f"seed_stability_summary_shard{SHARD}.csv")
    print(f"\nmean spread in errors across seeds: {agg['errors_sd'].mean():.2f}")
    print(f"largest single-model range: "
          f"{int((agg['errors_max'] - agg['errors_min']).max())} errors")
    print("\nInterpretation: any two systems whose mean error counts differ by less "
          "than roughly this spread cannot be ordered from a single run.")

           Model  n_seeds  errors_mean  errors_sd  errors_min  errors_max  wf1_mean  wf1_sd
BiomedBERT-large        5          3.8     1.6432           2           5    0.9888  0.0048
      ModernBERT        5          5.0     1.5811           3           7    0.9853  0.0047
      DeBERTa-v3        5          5.8     1.3038           4           7    0.9829  0.0038
Bio+ClinicalBERT        5          6.2     3.1937           4          11    0.9818  0.0093
table  -> /mnt/subedi-research/projects/bmo_rerun_2026/outputs/bert_models/seed_stability_summary_shard1.csv  (4 rows)

mean spread in errors across seeds: 1.93
largest single-model range: 7 errors

Interpretation: any two systems whose mean error counts differ by less than roughly this spread cannot be ordered from a single run.
